In [4]:
# task 1
import sys
import requests

POSTS_URL = 'https://jsonplaceholder.typicode.com/posts'
POSTS_COUNT = 5
TIMEOUT = 15


def fetch_posts(url: str = POSTS_URL, limit: int = POSTS_COUNT) -> list[dict]:
    response = requests.get(url, timeout=TIMEOUT)
    response.raise_for_status()
    posts = response.json()

    return posts[:limit]


def print_posts(posts: list[dict]) -> None:
    for number, post in enumerate(posts, start=1):
        print(f'Пост #{number} (id={post['id']})')
        print('-' * 60)
        print(f'Заголовок: {post['title']}')
        print('Тело:', '\n', post['body'], '\n')


def main() -> int:
    try:
        posts = fetch_posts()
    except requests.HTTPError as error:
        print(f'Сервер вернул ошибку: {error}', file=sys.stderr)
        return 1
    except requests.RequestException as error:
        print(f'Не удалось выполнить запрос: {error}', file=sys.stderr)
        return 1
    except ValueError:
        print('Ответ API не является корректным JSON', file=sys.stderr)
        return 1

    print_posts(posts)
    return 0


if __name__ == '__main__':
    exit_code = main()
    if exit_code:
        raise SystemExit(exit_code)


Пост #1 (id=1)
------------------------------------------------------------
Заголовок: sunt aut facere repellat provident occaecati excepturi optio reprehenderit
Тело: 
 quia et suscipit
suscipit recusandae consequuntur expedita et cum
reprehenderit molestiae ut ut quas totam
nostrum rerum est autem sunt rem eveniet architecto 

Пост #2 (id=2)
------------------------------------------------------------
Заголовок: qui est esse
Тело: 
 est rerum tempore vitae
sequi sint nihil reprehenderit dolor beatae ea dolores neque
fugiat blanditiis voluptate porro vel nihil molestiae ut reiciendis
qui aperiam non debitis possimus qui neque nisi nulla 

Пост #3 (id=3)
------------------------------------------------------------
Заголовок: ea molestias quasi exercitationem repellat qui ipsa sit aut
Тело: 
 et iusto sed quo iure
voluptatem occaecati omnis eligendi aut ad
voluptatem doloribus vel accusantium quis pariatur
molestiae porro eius odio et labore et velit aut 

Пост #4 (id=4)
---------------

In [8]:
# task 2
import os
import sys
import requests

WEATHER_URL = 'https://api.openweathermap.org/data/2.5/weather'
GEOCODING_URL = 'https://geocoding-api.open-meteo.com/v1/search'
OPEN_METEO_URL = 'https://api.open-meteo.com/v1/forecast'
TIMEOUT = 15

# Коды погоды WMO, которые возвращает Open-Meteo.
WEATHER_CODES = {
    0: 'ясно',
    1: 'преимущественно ясно',
    2: 'переменная облачность',
    3: 'пасмурно',
    45: 'туман',
    48: 'изморозь и туман',
    51: 'слабая морось',
    53: 'умеренная морось',
    55: 'сильная морось',
    56: 'слабая переохлаждённая морось',
    57: 'сильная переохлаждённая морось',
    61: 'небольшой дождь',
    63: 'умеренный дождь',
    65: 'сильный дождь',
    66: 'небольшой переохлаждённый дождь',
    67: 'сильный переохлаждённый дождь',
    71: 'небольшой снегопад',
    73: 'умеренный снегопад',
    75: 'сильный снегопад',
    77: 'снежные зёрна',
    80: 'небольшие ливни',
    81: 'умеренные ливни',
    82: 'сильные ливни',
    85: 'небольшой снегопад',
    86: 'сильный снегопад',
    95: 'гроза',
    96: 'гроза с небольшим градом',
    99: 'гроза с сильным градом',
}


class WeatherError(Exception):
    '''Ошибка, которую нужно показать пользователю без трейсбека.'''


def build_params(city: str, api_key: str) -> dict[str, str]:
    '''Формирует параметры запроса к OpenWeather.'''
    return {
        'q': city,
        'appid': api_key,
        'units': 'metric',
        'lang': 'ru',
    }


def get_weather(city: str, api_key: str) -> dict:
    '''Возвращает данные о текущей погоде в городе через OpenWeather.'''
    response = requests.get(
        WEATHER_URL,
        params=build_params(city, api_key),
        timeout=TIMEOUT,
    )

    if response.status_code == 401:
        raise WeatherError(
            'неверный API-ключ OpenWeather (проверьте OPENWEATHER_API_KEY)'
        )

    if response.status_code == 404:
        raise WeatherError(f'город «{city}» не найден')

    response.raise_for_status()

    return response.json()


def describe_weather_code(code: int | None) -> str:
    '''Переводит код погоды WMO в описание на русском.'''
    if code is None:
        return 'нет данных'

    return WEATHER_CODES.get(code, f'код погоды {code}')


def get_weather_open_meteo(city: str) -> dict:
    '''Возвращает погоду через Open-Meteo (резервный вариант без ключа).'''
    geo_response = requests.get(
        GEOCODING_URL,
        params={'name': city, 'count': 1, 'language': 'ru', 'format': 'json'},
        timeout=TIMEOUT,
    )
    geo_response.raise_for_status()
    results = geo_response.json().get('results') or []
    if not results:
        raise WeatherError(f'город «{city}» не найден')

    place = results[0]

    weather_response = requests.get(
        OPEN_METEO_URL,
        params={
            'latitude': place['latitude'],
            'longitude': place['longitude'],
            'current': 'temperature_2m,weather_code',
            'timezone': 'auto',
        },
        timeout=TIMEOUT,
    )
    weather_response.raise_for_status()
    current = weather_response.json().get('current') or {}
    temperature = current.get('temperature_2m')
    if temperature is None:
        raise WeatherError('сервис погоды не вернул температуру')

    # Приводим ответ к тому же виду, что и у OpenWeather.
    return {
        'name': place.get('name') or city,
        'main': {'temp': temperature},
        'weather': [
            {'description': describe_weather_code(current.get('weather_code'))}
        ],
    }


def print_weather(weather: dict) -> None:
    '''Выводит температуру и описание погоды.'''
    city = weather.get('name') or 'неизвестно'
    temperature = weather.get('main', {}).get('temp')
    descriptions = weather.get('weather') or [{}]
    description = descriptions[0].get('description') or 'нет данных'

    print(f'Город: {city}')
    print('Температура: нет данных') if temperature is None else print(f'Температура: {temperature:+.1f}C')
    print(f'Описание погоды: {description}')


api_key = os.getenv('OPENWEATHER_API_KEY', '').strip()
city = input('Введите название города: ').strip()
if not city:
    print('Название города не может быть пустым', file=sys.stderr)
else:
    try:
        if api_key:
            weather = get_weather(city, api_key)
        else:
            print(
                'API-ключ OpenWeather не задан — использую Open-Meteo (без ключа).',
                file=sys.stderr,
            )
            weather = get_weather_open_meteo(city)
        print_weather(weather)
    except WeatherError as error:
        print(f'Ошибка: {error}', file=sys.stderr)
    except requests.RequestException as error:
        print(f'Не удалось выполнить запрос: {error}', file=sys.stderr)
    except ValueError:
        print('Ответ API не является корректным JSON', file=sys.stderr)


Введите название города:  Moscow


API-ключ OpenWeather не задан — использую Open-Meteo (без ключа).


Город: Москва
Температура: +4.1 °C
Описание погоды: небольшой дождь
